In [2]:
import sys

import pandas as pd
import logging
import pathlib
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm import tqdm

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger("testing")

In [3]:
TIME_COL = "xltime"

# LOB columns
LOB_COLUMNS = []
for level in range(1, 11):
    LOB_COLUMNS += [
        f"bid-price-{level}",
        f"bid-volume-{level}",
        f"ask-price-{level}",
        f"ask-volume-{level}",
    ]


def fix_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Fix the column-header misalignment present in raw files.

    Two variants are observed across the raw files:

    - 42-column files (CSV + some parquets): a spurious second column
      ('Unnamed: 1' / 'V2') is inserted after 'xltime', shifting all LOB
      names one position right and leaving the last column NaN/garbage.
      Fix: drop the last column, then rename all 41.

    - 41-column files (some parquets): the parquet was written with the
      first data row used as column headers (data values as names, e.g.
      '42738.21...', '46.49', …).  No extra column — just wrong names.
      Fix: rename all 41 columns directly.

    If the DataFrame already has the correct column names, it is returned
    unchanged.
    """
    new_names = [TIME_COL] + LOB_COLUMNS

    if list(df.columns) == new_names:
        return df

    if len(df.columns) == len(new_names) + 1:
        df = df.iloc[:, :-1].copy()
        df.columns = new_names
    elif len(df.columns) == len(new_names):
        df = df.copy()
        df.columns = new_names
    else:
        logger.warning(
            "Unexpected column count %d (expected %d or %d). Skipping column fix.",
            len(df.columns), len(new_names), len(new_names) + 1,
        )
    return df

In [ ]:
df = pd.read_parquet('../data/raw/TOTF.PA-book/2009-05-21-TOTF.PA-book.parquet')
df = fix_columns(df)
df.head()

,xltime,bid-price-1,bid-volume-1,ask-price-1,ask-volume-1,bid-price-2,bid-volume-2,ask-price-2,ask-volume-2,bid-price-3,...,ask-price-8,ask-volume-8,bid-price-9,bid-volume-9,ask-price-9,ask-volume-9,bid-price-10,bid-volume-10,ask-price-10,ask-volume-10
0,39954.167007,41.10,100.0,41.32,235.0,(),(),(),(),(),...,(),(),(),(),(),(),(),(),(),()
1,39954.218779,41.20,84.0,41.32,235.0,(),(),(),(),(),...,(),(),(),(),(),(),(),(),(),()
2,39954.218790,41.32,38.0,41.32,235.0,(),(),(),(),(),...,(),(),(),(),(),(),(),(),(),()
3,39954.218796,41.32,38.0,41.10,50.0,(),(),(),(),(),...,(),(),(),(),(),(),(),(),(),()
4,39954.218796,41.20,50.0,41.10,50.0,(),(),(),(),(),...,(),(),(),(),(),(),(),(),(),()


In [5]:
depth_cols = [c for c in df.columns
              if c.startswith(('bid-price-', 'bid-volume-',
                               'ask-price-', 'ask-volume-'))
              and int(c.rsplit('-', 1)[-1]) > 1]
depth_cols.sort()


In [6]:
print(df[depth_cols].sum(numeric_only=True).sum(numeric_only=True))

0.0


In [ ]:
from pathlib import Path
import numpy as np

root = Path("../data/raw")
files_2009 = sorted(root.glob("*-book/2009-*-*-*-book.parquet"))

records = []
for fp in tqdm(files_2009, desc="Scanning 2009 files"):
    try:
        dfi = pd.read_parquet(fp)
        dfi = fix_columns(dfi)

        depth_cols_i = [
            c for c in dfi.columns
            if c.startswith(("bid-price-", "bid-volume-", "ask-price-", "ask-volume-"))
            and int(c.rsplit("-", 1)[-1]) > 1
        ]

        if not depth_cols_i:
            records.append({
                "file": fp.name,
                "rows": len(dfi),
                "depth_cols": 0,
                "empty_tuple_cells": np.nan,
                "total_depth_cells": np.nan,
                "pct_empty_tuple_cells": np.nan,
                "all_depth_cells_empty_tuple": False,
                "any_depth_tuple": False,
                "error": "No depth columns found",
            })
            continue

        depth_df = dfi[depth_cols_i]
        # Fast check: tuple() values are rendered as the string "()".
        empty_tuple_mask = depth_df.astype(str).eq("()")
        empty_tuple_cells = int(empty_tuple_mask.to_numpy().sum())
        total_depth_cells = int(depth_df.shape[0] * depth_df.shape[1])
        pct_empty = empty_tuple_cells / total_depth_cells if total_depth_cells else np.nan
        all_empty = bool(empty_tuple_cells == total_depth_cells and total_depth_cells > 0)
        any_empty = bool(empty_tuple_cells > 0)

        records.append({
            "file": fp.name,
            "rows": len(dfi),
            "depth_cols": len(depth_cols_i),
            "empty_tuple_cells": empty_tuple_cells,
            "total_depth_cells": total_depth_cells,
            "pct_empty_tuple_cells": pct_empty,
            "all_depth_cells_empty_tuple": all_empty,
            "any_depth_tuple": any_empty,
            "error": "",
        })
    except Exception as e:
        records.append({
            "file": fp.name,
            "rows": np.nan,
            "depth_cols": np.nan,
            "empty_tuple_cells": np.nan,
            "total_depth_cells": np.nan,
            "pct_empty_tuple_cells": np.nan,
            "all_depth_cells_empty_tuple": False,
            "any_depth_tuple": False,
            "error": str(e),
        })

summary_df = pd.DataFrame(records).sort_values("file").reset_index(drop=True)

n_files = len(summary_df)
n_all_empty = int(summary_df["all_depth_cells_empty_tuple"].sum())
n_any_empty = int(summary_df["any_depth_tuple"].sum())
n_errors = int((summary_df["error"] != "").sum())

print(f"2009 files scanned: {n_files}")
print(f"Files with all depth>1 cells equal to ()/\"()\": {n_all_empty}")
print(f"Files with at least one ()/\"()\" in depth>1: {n_any_empty}")
print(f"Files with read/processing errors: {n_errors}")

summary_df.head(20)

Scanning 2009 files:   0%|          | 0/261 [00:00<?, ?it/s]

Scanning 2009 files:  26%|██▌       | 67/261 [01:25<04:07,  1.27s/it]


KeyboardInterrupt: 